# Semantic Search on large text


## Objective
```
Long Document
     ↓
Split into chunks
     ↓
Convert each chunk → embedding
     ↓
Store in Pinecone
     ↓
Query → retrieve relevant chunks


# Chunking into fixed size

# Long text

In [42]:
# Note: 2) The following long text contains Biology + Quantum Mechanics 
# 1) I added some extra stuff toward the end which is not true

long_text = """
Biology:

Biology is the study of life and living organisms, covering everything from the tiny chemical reactions inside
a cell to the complex interactions of entire ecosystems. It explores how creatures grow, reproduce, and 
evolve to survive in an ever-changing world. By understanding biology, we get a front-row seat to the 
understanding of the natural world and our own bodies. 
Here are some fascinating facts that show how wild nature can be:

Human Cloud: You are currently surrounded by a "microbial cloud" of millions of bacteria that is unique to you, 
much like a fingerprint.

Tree Lungs: A single mature leafy tree can produce enough oxygen in one season to support 10 people for a year.

Immortal Jellyfish: The Turritopsis dohrnii can technically live forever by resetting its cells back to 
their earliest stage when threatened or injured.

Shared Ancestry: Humans share about 60% of their DNA with bananas.

Fast DNA: If you uncoiled all the DNA in your body and ran it end-to-end, it would stretch from Earth to Pluto and back—twice.


Quantum Mechanics: A Grand Tour of the Tiny Universe
Executive Summary: Quantum mechanics is the branch of physics governing the very small – atoms, electrons, 
photons, and other particles. As one source notes, “Quantum mechanics is the fundamental physical theory 
that describes the behavior of matter and of light; its unusual characteristics typically occur…at and 
below the scale of atoms.”[1]. In other words, it’s how the universe works when you zoom in far beyond 
everyday experience. It arose in the early 1900s to explain puzzling experiments (like blackbody radiation
and the photoelectric effect) that classical physics couldn’t[2]. This report gives a concise introduction:
we’ll define quantum mechanics, outline its history with a timeline, meet two pioneers 
(Max Planck and Erwin Schrödinger) via short bios and a comparison table, then unpack essential quantum 
concepts (wave–particle duality, superposition, uncertainty, quantization, quantum states/operators, 
measurement, entanglement, spin and Pauli exclusion, tunneling) with simple analogies. We’ll then
glance at modern applications (quantum computing, cryptography, sensors) and open questions (interpretation 
puzzles, quantum gravity). Throughout, the tone is clear and accessible (with a bit of humor) and citations
to reliable sources ensure accuracy.
What is Quantum Mechanics?
At its core, quantum mechanics is the rules of the microscopic world. Classical physics (Newton, Maxwell, etc.) 
works for cars and planets, but fails spectacularly for atoms and light on tiny scales[1][3]. 
Quantum mechanics tells us that energy comes in discrete packets (“quanta”), particles behave like waves, 
and outcomes are fundamentally probabilistic. In plain language, you can’t predict exactly where an 
electron is or what it will do until you measure it – only the odds. As one general-audience summary 
puts it, quantum physics “studies how matter and energy behave at very small, microscopic levels”[3].
For example, electrons in an atom don’t orbit like planets; instead, their location is described by 
a “cloud” or wave function that gives probabilities.
The mathematics (wave functions, operators, etc.) is rich, but the core ideas are about granularity 
(all energy comes in chunks) and fuzziness (nature is not fully deterministic at small scales)[4]. 
We’ll see that quantum mechanics underpins everything from the colors of atoms to the operation of 
lasers and silicon chips. First, let’s step back and see how these ideas were discovered.

Current research as of this year is focused on reconciling quantum mechanics with general relativity and with 
arts and humanities.
"""

In [43]:
def chunk_text(text, chunk_size=100):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks

chunks = chunk_text(long_text, chunk_size=50)

print("Number of chunks:", len(chunks))
# print("Chunks:\n", chunks)

for c in chunks:
    print(c)
    print("------")

Number of chunks: 12
Biology: Biology is the study of life and living organisms, covering everything from the tiny chemical reactions inside a cell to the complex interactions of entire ecosystems. It explores how creatures grow, reproduce, and evolve to survive in an ever-changing world. By understanding biology, we get a front-row seat to
------
the understanding of the natural world and our own bodies. Here are some fascinating facts that show how wild nature can be: Human Cloud: You are currently surrounded by a "microbial cloud" of millions of bacteria that is unique to you, much like a fingerprint. Tree Lungs: A single mature
------
leafy tree can produce enough oxygen in one season to support 10 people for a year. Immortal Jellyfish: The Turritopsis dohrnii can technically live forever by resetting its cells back to their earliest stage when threatened or injured. Shared Ancestry: Humans share about 60% of their DNA with bananas. Fast
------
DNA: If you uncoiled all the DNA in y

In [ ]:
# Following too long time
!pip install pinecone sentence-transformers

# SO I used 
# !python -m !pip install pinecone sentence-transformers

In [6]:
# Import Libraries: Took a 4-5 minutes

from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

In [7]:
# Need API Key
pc = Pinecone(api_key="pcsk_API_KEY_HERE")

# Create an index

In [26]:
index_name = "genai-demo-large-text"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

# Connect to Index

In [27]:
index = pc.Index(index_name)

In [40]:
vectors = [
    {
        "_id": str(i),
        "chunk_text": doc   # must match field_map
    }
    for i, doc in enumerate(chunks)
]

# print(vectors)
for v in vectors:
    print(v)
    print("-------")

{'_id': '0', 'chunk_text': 'Quantum Mechanics: A Grand Tour of the Tiny Universe Executive Summary: Quantum mechanics is the branch of physics governing the very small – atoms, electrons, photons, and other particles. As one source notes, “Quantum mechanics is the fundamental physical theory that describes the behavior of matter and of light; its'}
-------
{'_id': '1', 'chunk_text': 'unusual characteristics typically occur…at and below the scale of atoms.”[1]. In other words, it’s how the universe works when you zoom in far beyond everyday experience. It arose in the early 1900s to explain puzzling experiments (like blackbody radiation and the photoelectric effect) that classical physics couldn’t[2]. This report gives'}
-------
{'_id': '2', 'chunk_text': 'a concise introduction: we’ll define quantum mechanics, outline its history with a timeline, meet two pioneers (Max Planck and Erwin Schrödinger) via short bios and a comparison table, then unpack essential quantum concepts (wave–part

# Store the documents in vector DB

In [29]:
# Upsert the records into a namespace
index.upsert_records("example-namespace", vectors)

UpsertResponse(upserted_count=8, _response_info={'raw_headers': {'date': 'Thu, 19 Mar 2026 15:44:47 GMT', 'content-length': '0', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-api-version': '2025-10', 'x-envoy-upstream-service-time': '600', 'x-pinecone-response-duration-ms': '601', 'server': 'envoy'}})

In [30]:
# Wait for the upserted vectors to be indexed
import time
time.sleep(10)

# View stats for the index
stats = index.describe_index_stats()
print(stats)

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '188',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 19 Mar 2026 15:45:03 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '37',
                                    'x-pinecone-request-latency-ms': '36',
                                    'x-pinecone-response-duration-ms': '39'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'example-namespace': {'vector_count': 8}},
 'storageFullness': 0.0,
 'total_vector_count': 8,
 'vector_type': 'dense'}


# Perform Semantic Search

In [31]:
# Define the query
query = "What is the latest research on quantum mechanic focused on as of this year ?"

# Search the dense index
results = index.search(
    namespace="example-namespace",
    query={
        "top_k": 4,
        "inputs": {
            'text': query
        }
    }
)

In [32]:
# Returns results sorted by similarity score
print(results)

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '1659',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 19 Mar 2026 15:45:11 GMT',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '304',
                                    'x-pinecone-api-version': '2025-10',
                                    'x-pinecone-max-indexed-lsn': '1',
                                    'x-pinecone-response-duration-ms': '306'}},
 'result': {'hits': [{'_id': '7',
                      '_score': 0.4420064687728882,
                      'fields': {'chunk_text': 'atoms to the operation of '
                                               'lasers and silicon chips. '
                                               'First, let’s step back and see '
                                               'how

In [41]:
# Print the results
for hit in results['result']['hits']: # returns result sorted by score
    # print(hit)
    print(f"id: {hit['_id']:<5} | score: {round(hit['_score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")
    print("----")

id: 7     | score: 0.44  | text: atoms to the operation of lasers and silicon chips. First, let’s step back and see how these ideas were discovered. Current research as of this year is focused on reconciling quantum mechanics with general relativity and with arts and humanities
----
id: 3     | score: 0.26  | text: We’ll then glance at modern applications (quantum computing, cryptography, sensors) and open questions (interpretation puzzles, quantum gravity). Throughout, the tone is clear and accessible (with a bit of humor) and citations to reliable sources ensure accuracy. What is Quantum Mechanics? At its core, quantum mechanics is the rules of the
----
id: 0     | score: 0.23  | text: Quantum Mechanics: A Grand Tour of the Tiny Universe Executive Summary: Quantum mechanics is the branch of physics governing the very small – atoms, electrons, photons, and other particles. As one source notes, “Quantum mechanics is the fundamental physical theory that describes the behavior of matter 

# Observation:
- The query matches with the first hit.
- The results do not have any content related to Biology.

# Delete the index when done with Demo

In [34]:
pc.delete_index(index_name)

# (EXTRA) Overlapping chunks

In [44]:
# 1)
def chunk_text_overlap(text, chunk_size=50, overlap=10):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks


In [45]:
# 2) 
chunks = chunk_text_overlap(long_text, chunk_size=50, overlap=5)

for c in chunks:
    print(c)
    print("------")


Biology: Biology is the study of life and living organisms, covering everything from the tiny chemical reactions inside a cell to the complex interactions of entire ecosystems. It explores how creatures grow, reproduce, and evolve to survive in an ever-changing world. By understanding biology, we get a front-row seat to
------
get a front-row seat to the understanding of the natural world and our own bodies. Here are some fascinating facts that show how wild nature can be: Human Cloud: You are currently surrounded by a "microbial cloud" of millions of bacteria that is unique to you, much like a fingerprint.
------
you, much like a fingerprint. Tree Lungs: A single mature leafy tree can produce enough oxygen in one season to support 10 people for a year. Immortal Jellyfish: The Turritopsis dohrnii can technically live forever by resetting its cells back to their earliest stage when threatened or injured. Shared Ancestry:
------
threatened or injured. Shared Ancestry: Humans share about 